# Smart Autocomplete ML Demo

This notebook validates the Deep Learning logic for next-word prediction using PyTorch (LSTM).
We use a small fallback dataset (a public text file) to demonstrate the logic before modularizing the code.

## 1. Data Loading
We will download a small public text dataset (e.g., a tiny Shakespeare snippet) directly to avoid Kaggle CLI setup.

In [ ]:
import urllib.request
import re
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
file_path = 'dataset.txt'
urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()[:50000] # Use only the first 50k characters for fast demo
print(f"Loaded {len(raw_text)} characters.")

## 2. Preprocessing
Lowercasing, removing punctuation, and tokenizing.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z \n]', '', text)
    return text.split()

words = clean_text(raw_text)
print(f"Total words: {len(words)}")
print(f"Sample words: {words[:10]}")

## 3. Vocabulary Creation

In [ ]:
word_counts = Counter(words)
vocab = {"<pad>": 0, "<unk>": 1}
idx = 2
for w, c in word_counts.items():
    if c >= 2: # Min freq filter
        vocab[w] = idx
        idx += 1

idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

## 4. Sequence Generation
Create input-target pairs (e.g., 5 words predict the 6th).

In [ ]:
SEQ_LENGTH = 5
sequences = []
targets = []

tokens = [vocab.get(w, vocab["<unk>"]) for w in words]
for i in range(len(tokens) - SEQ_LENGTH):
    sequences.append(tokens[i:i+SEQ_LENGTH])
    targets.append(tokens[i+SEQ_LENGTH])

X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

print(f"Generated sequences: {X.shape}, Targets: {y.shape}")

## 5. Model Building (LSTM)

In [ ]:
class AutocompleteLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super(AutocompleteLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        out = self.embedding(x)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

model = AutocompleteLSTM(vocab_size)
print(model)

## 6. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
dataset = torch.utils.data.TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

epochs = 3
model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

## 7. Prediction Function

In [ ]:
def predict_next(text, top_k=3):
    model.eval()
    words = clean_text(text)
    tokens = [vocab.get(w, vocab["<unk>"]) for w in words[-SEQ_LENGTH:]]
    while len(tokens) < SEQ_LENGTH:
        tokens.insert(0, vocab["<pad>"])
    
    inputs = torch.tensor([tokens], dtype=torch.long)
    with torch.no_grad():
        output = model(inputs)
        probs = torch.softmax(output, dim=1).squeeze()
        top_probs, top_idx = torch.topk(probs, top_k)
        
    return [idx2word.get(i.item(), '') for i in top_idx]


## 8. Example Outputs

In [ ]:
test_phrases = [
    "it is a",
    "what do you",
    "i have to"
]
for phrase in test_phrases:
    suggestions = predict_next(phrase)
    print(f"Input: '{phrase}' -> Predicted next words: {suggestions}")